# Medical MNIST Classification with PyTorch

의료 이미지 6개 클래스를 CNN으로 분류하는 프로젝트입니다.

**데이터셋:** [Medical MNIST](https://www.kaggle.com/datasets/andrewmvd/medical-mnist)  
**클래스:** AbdomenCT, BreastMRI, ChestCT, CXR, Hand, HeadCT  
**이미지:** 64x64 grayscale, 총 58,954장  

## 목차
1. 환경 설정 & 라이브러리 import
2. 데이터 탐색 (EDA)
3. Dataset & DataLoader 구성
4. CNN 모델 정의
5. 학습 (Training)
6. 평가 (Evaluation) & 시각화

---
## 1. 환경 설정 & 라이브러리 import

PyTorch와 필요한 라이브러리를 불러옵니다.  
GPU가 사용 가능하면 자동으로 GPU를 사용합니다.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm

# GPU 사용 가능 여부 확인
# - 'cuda': NVIDIA GPU 사용
# - 'cpu': GPU 없으면 CPU 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 재현성을 위한 랜덤 시드 고정
# 같은 시드 = 같은 결과 (실험 재현 가능)
torch.manual_seed(42)
np.random.seed(42)

---
## 2. 데이터 탐색 (EDA)

EDA = Exploratory Data Analysis (탐색적 데이터 분석)  
모델을 만들기 전에 데이터가 어떻게 생겼는지 먼저 확인합니다.

확인할 것:
- 폴더 구조 (클래스별 폴더)
- 클래스당 이미지 수 (불균형 여부)
- 실제 이미지 샘플

In [ ]:
# Kaggle Notebook에서의 데이터 경로
# Kaggle에서 "Add Data"로 medical-mnist를 추가하면 이 경로에 자동 마운트됩니다
DATA_DIR = '/kaggle/input/medical-mnist'

# 폴더 구조 확인: 각 폴더 = 하나의 클래스
classes = sorted(os.listdir(DATA_DIR))
print(f'클래스 목록 ({len(classes)}개): {classes}')

# 클래스별 이미지 수 세기
class_counts = {}
for cls in classes:
    cls_path = os.path.join(DATA_DIR, cls)
    if os.path.isdir(cls_path):
        count = len(os.listdir(cls_path))
        class_counts[cls] = count
        print(f'  {cls}: {count}장')

print(f'\n총 이미지 수: {sum(class_counts.values())}장')

### 클래스 분포 시각화

클래스 간 이미지 수가 균등한지 확인합니다.  
불균형이 심하면 모델이 다수 클래스에 편향될 수 있습니다.

In [ ]:
plt.figure(figsize=(10, 5))
bars = plt.bar(class_counts.keys(), class_counts.values(), color='steelblue')
plt.title('Class Distribution (클래스별 이미지 수)', fontsize=14)
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=45)

# 각 막대 위에 숫자 표시
for bar, count in zip(bars, class_counts.values()):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 100,
             str(count), ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

### 샘플 이미지 확인

각 클래스에서 이미지 2장씩 가져와서 실제 모습을 봅니다.

In [ ]:
fig, axes = plt.subplots(2, len(classes), figsize=(18, 6))
fig.suptitle('Sample Images per Class (클래스별 샘플)', fontsize=14)

for col, cls in enumerate(classes):
    cls_path = os.path.join(DATA_DIR, cls)
    img_files = os.listdir(cls_path)[:2]  # 첫 2장
    
    for row, img_file in enumerate(img_files):
        img = Image.open(os.path.join(cls_path, img_file))
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(cls, fontsize=11)

plt.tight_layout()
plt.show()

# 이미지 크기 확인
sample_img = Image.open(os.path.join(DATA_DIR, classes[0], os.listdir(os.path.join(DATA_DIR, classes[0]))[0]))
print(f'이미지 크기: {sample_img.size}, 모드: {sample_img.mode}')

---
## 3. Dataset & DataLoader 구성

PyTorch에서 데이터를 다루는 핵심 구조:

- **Dataset**: 데이터 하나를 가져오는 방법을 정의 (이미지 읽기 + 라벨 반환)
- **DataLoader**: Dataset에서 배치(batch) 단위로 데이터를 꺼내주는 역할

```
전체 데이터 → Dataset (개별 접근) → DataLoader (배치 단위로 묶어서 전달) → 모델
```

### 데이터 분할
- **Train (70%)**: 모델이 학습하는 데이터
- **Validation (15%)**: 학습 중간에 성능을 확인하는 데이터 (학습에 사용 안 함)
- **Test (15%)**: 최종 성능을 측정하는 데이터 (학습과 검증에 모두 사용 안 함)

In [ ]:
class MedicalMNISTDataset(Dataset):
    """
    Medical MNIST 이미지를 PyTorch Dataset으로 만드는 클래스.
    
    __len__: 전체 이미지 수 반환
    __getitem__: 인덱스로 (이미지, 라벨) 쌍 반환
    """
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        # 모든 이미지 경로와 라벨을 미리 수집
        self.samples = []
        for cls in self.classes:
            cls_path = os.path.join(data_dir, cls)
            if not os.path.isdir(cls_path):
                continue
            for img_name in os.listdir(cls_path):
                img_path = os.path.join(cls_path, img_name)
                self.samples.append((img_path, self.class_to_idx[cls]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        # 이미지를 grayscale로 읽기
        image = Image.open(img_path).convert('L')
        
        # transform 적용 (텐서 변환, 정규화 등)
        if self.transform:
            image = self.transform(image)
        
        return image, label

### Transform (전처리) 정의

이미지를 모델에 넣기 전에 변환하는 과정:
- `ToTensor()`: PIL 이미지 → PyTorch 텐서 (0~255 → 0.0~1.0)
- `Normalize()`: 평균 0.5, 표준편차 0.5로 정규화 → 값 범위가 -1 ~ 1이 됨

정규화를 하면 모델이 더 안정적으로 학습합니다.

In [ ]:
# 이미지 전처리 파이프라인
transform = transforms.Compose([
    transforms.Resize((64, 64)),    # 크기 통일 (이미 64x64이지만 안전장치)
    transforms.ToTensor(),           # 텐서로 변환 (0~1)
    transforms.Normalize([0.5], [0.5])  # 정규화 (-1~1)
])

# 전체 데이터셋 생성
full_dataset = MedicalMNISTDataset(DATA_DIR, transform=transform)
print(f'전체 데이터셋 크기: {len(full_dataset)}')
print(f'클래스 매핑: {full_dataset.class_to_idx}')

# Train / Validation / Test 분할 (70% / 15% / 15%)
total = len(full_dataset)
train_size = int(0.7 * total)
val_size = int(0.15 * total)
test_size = total - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)  # 분할 재현성
)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

### DataLoader 생성

DataLoader는 데이터를 **배치(batch)** 단위로 묶어서 모델에 전달합니다.

- `batch_size=64`: 한 번에 64장씩 처리
- `shuffle=True`: 매 epoch마다 데이터 순서를 섞음 (과적합 방지)
- `num_workers=2`: 데이터 로딩을 병렬로 처리 (속도 향상)

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# 배치 하나 확인
images, labels = next(iter(train_loader))
print(f'배치 이미지 shape: {images.shape}')  # [64, 1, 64, 64] = [배치, 채널, 높이, 너비]
print(f'배치 라벨 shape: {labels.shape}')     # [64]

---
## 4. CNN 모델 정의

CNN (Convolutional Neural Network)은 이미지 인식에 특화된 신경망입니다.

### 구조 설명
```
입력 (1x64x64)
  ↓
Conv2d → ReLU → MaxPool  (특징 추출 1단계)
  ↓
Conv2d → ReLU → MaxPool  (특징 추출 2단계)
  ↓
Conv2d → ReLU → MaxPool  (특징 추출 3단계)
  ↓
Flatten → FC → ReLU → Dropout → FC  (분류)
  ↓
출력 (6개 클래스 확률)
```

- **Conv2d**: 이미지에서 패턴(엣지, 질감 등)을 감지하는 필터
- **ReLU**: 활성화 함수 — 음수를 0으로 만들어 비선형성 추가
- **MaxPool**: 이미지 크기를 절반으로 줄여서 계산량 감소 + 위치 불변성
- **Dropout**: 학습 중 뉴런 일부를 랜덤으로 끄기 → 과적합 방지
- **FC (Fully Connected)**: 추출된 특징을 기반으로 최종 분류

In [ ]:
class MedicalCNN(nn.Module):
    def __init__(self, num_classes=6):
        super(MedicalCNN, self).__init__()
        
        # 특징 추출부 (Convolutional Layers)
        self.features = nn.Sequential(
            # 1단계: 1채널 → 32채널, 3x3 필터
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 64x64 → 32x32
            
            # 2단계: 32채널 → 64채널
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 32x32 → 16x16
            
            # 3단계: 64채널 → 128채널
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 16x16 → 8x8
        )
        
        # 분류부 (Fully Connected Layers)
        self.classifier = nn.Sequential(
            nn.Flatten(),               # 128x8x8 = 8192 → 1차원 벡터
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.5),            # 50% 확률로 뉴런 끄기
            nn.Linear(256, num_classes)  # 최종 6개 클래스 출력
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 모델 생성 & GPU로 이동
model = MedicalCNN(num_classes=len(classes)).to(device)
print(model)

# 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n총 파라미터: {total_params:,} | 학습 가능: {trainable_params:,}')

---
## 5. 학습 (Training)

### 학습에 필요한 3가지
1. **Loss Function (손실 함수)**: 모델의 예측이 정답과 얼마나 다른지 측정
   - `CrossEntropyLoss`: 다중 클래스 분류의 표준 손실 함수
2. **Optimizer (최적화기)**: 손실을 줄이는 방향으로 모델 파라미터를 업데이트
   - `Adam`: 가장 널리 쓰이는 최적화 알고리즘
3. **Epochs**: 전체 데이터를 몇 번 반복 학습할지

### 학습 루프
```
매 epoch마다:
  1. Train: 배치 단위로 → 예측 → 손실 계산 → 역전파 → 파라미터 업데이트
  2. Validate: 학습 안 하고 성능만 확인 (과적합 모니터링)
```

In [ ]:
# 하이퍼파라미터 설정
NUM_EPOCHS = 10
LEARNING_RATE = 0.001

# 손실 함수 & 최적화기
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 학습 기록 저장용
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """1 epoch 학습 수행"""
    model.train()  # 학습 모드 (Dropout 활성화)
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()       # 1) 이전 기울기 초기화
        outputs = model(images)     # 2) 순전파 (예측)
        loss = criterion(outputs, labels)  # 3) 손실 계산
        loss.backward()             # 4) 역전파 (기울기 계산)
        optimizer.step()            # 5) 파라미터 업데이트
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    """검증/테스트 수행 (학습 없이 성능만 측정)"""
    model.eval()  # 평가 모드 (Dropout 비활성화)
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # 기울기 계산 비활성화 (메모리 절약)
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

### 학습 실행

10 epoch 동안 학습합니다.  
매 epoch마다 train loss/accuracy와 validation loss/accuracy를 출력합니다.

In [ ]:
print('학습 시작!')
print('=' * 70)

best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    # 기록 저장
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Best 모델 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
    
    print(f'Epoch [{epoch+1:2d}/{NUM_EPOCHS}] '
          f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}'
          f'{" ★ Best" if val_acc >= best_val_acc else ""}')

print('=' * 70)
print(f'학습 완료! Best Validation Accuracy: {best_val_acc:.4f}')

### 학습 곡선 시각화

Loss와 Accuracy가 epoch에 따라 어떻게 변하는지 그래프로 확인합니다.

- **Train과 Val이 같이 개선**: 정상 학습
- **Train은 개선되는데 Val은 멈춤/악화**: 과적합 (overfitting)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

# Loss 그래프
ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs_range, history['val_loss'], 'r-o', label='Val Loss')
ax1.set_title('Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy 그래프
ax2.plot(epochs_range, history['train_acc'], 'b-o', label='Train Acc')
ax2.plot(epochs_range, history['val_acc'], 'r-o', label='Val Acc')
ax2.set_title('Accuracy per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. 평가 (Evaluation) & 시각화

학습이 끝난 모델을 **Test 데이터**로 최종 평가합니다.  
Test 데이터는 학습에 한 번도 사용되지 않았으므로, 모델의 진짜 성능을 보여줍니다.

In [ ]:
# Best 모델 로드
model.load_state_dict(torch.load('best_model.pth'))

# Test 데이터로 최종 평가
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'\n===== Test Results =====')
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)')

### Confusion Matrix (혼동 행렬)

각 클래스가 어떤 클래스로 분류되었는지 보여주는 표입니다.
- 대각선: 정확하게 분류된 수
- 대각선 외: 잘못 분류된 수 (어떤 클래스끼리 혼동하는지 파악)

In [ ]:
# 모든 예측값과 정답 수집
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Confusion Matrix 계산 & 시각화
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# 클래스별 상세 리포트
print('\n===== Classification Report =====')
print(classification_report(all_labels, all_preds, target_names=classes))

### 샘플 예측 시각화

테스트 데이터에서 랜덤으로 이미지를 뽑아서 모델의 예측 결과를 확인합니다.
- 초록색: 정답
- 빨간색: 오답

In [ ]:
# 테스트 세트에서 랜덤 16장 시각화
fig, axes = plt.subplots(4, 4, figsize=(12, 12))

# 랜덤 인덱스
indices = np.random.choice(len(test_dataset), 16, replace=False)

model.eval()
with torch.no_grad():
    for i, idx in enumerate(indices):
        image, label = test_dataset[idx]
        
        # 모델 예측
        output = model(image.unsqueeze(0).to(device))
        _, predicted = output.max(1)
        pred_idx = predicted.item()
        
        # 시각화
        ax = axes[i // 4, i % 4]
        # 정규화 역변환 (시각화용)
        img_display = image.squeeze().numpy() * 0.5 + 0.5
        ax.imshow(img_display, cmap='gray')
        
        actual = classes[label]
        pred = classes[pred_idx]
        color = 'green' if label == pred_idx else 'red'
        ax.set_title(f'Pred: {pred}\nActual: {actual}', color=color, fontsize=9)
        ax.axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()

---
## Summary

이 노트북에서 수행한 작업:

1. **EDA**: 6개 의료 이미지 클래스, 약 59,000장, 64x64 grayscale 확인
2. **데이터 준비**: Custom Dataset, Train/Val/Test 분할, DataLoader 구성
3. **모델**: 3-layer CNN (Conv→ReLU→Pool) + FC classifier
4. **학습**: 10 epochs, Adam optimizer, CrossEntropyLoss
5. **평가**: Test accuracy, Confusion Matrix, 샘플 예측 시각화

### 다음 단계 (Next Project)
- Transfer Learning (사전 학습 모델 활용: ResNet, EfficientNet 등)
- Data Augmentation (회전, 반전 등으로 데이터 다양성 확보)
- 더 복잡한 데이터셋 도전 (Breast Cancer Histopathology 등)